In [11]:
import urllib.parse
import urllib.request
import requests
import xml.etree.ElementTree as ET
from io import BytesIO
from PyPDF2 import PdfReader
import os
import time
import re
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle
import ollama

In [12]:
client = ollama.Client()

model_name = "llama3"

In [72]:
sentence_transformer_model =  SentenceTransformer('all-MiniLM-L6-v2')

dimension = 384 # the minilm model has an output dimension of 384
doc_index = faiss.IndexFlatL2(dimension)
chunk_index = faiss.IndexFlatL2(dimension)

In [63]:
# question = input("Enter your question: ")
question = "If the Roman Empire had never fallen, how might the development of Western legal traditions, specifically concerning the rights of individuals versus the state, have differed from the actual historical trajectory?"

In [15]:
# Option 1 - Using the question directly as the keyword.
keyword = question

num_articles = 100
encodingmethod = "utf-8"
errortype = "strict"

In [16]:
encoded_search_term = urllib.parse.quote(keyword, encoding=encodingmethod, errors=errortype)
url = f'http://export.arxiv.org/api/query?search_query=all:{encoded_search_term}&start=0&max_results={num_articles}'

print(f"Searching for '{keyword}' on arXiv...")
print(f"URL: {url}")

try:
    response = urllib.request.urlopen(url)
    try:
        url_read = response.read().decode("utf-8")
    except UnicodeDecodeError:
        response = urllib.request.urlopen(url)
        url_read = response.read().decode("utf-8", errors="ignore")

    parse_xml = ET.fromstring(url_read)
    print("Successfully retrieved search results!")
except Exception as e:
    print(f"Error retrieving data: {e}")
    raise

Searching for 'If the Roman Empire had never fallen, how might the development of Western legal traditions, specifically concerning the rights of individuals versus the state, have differed from the actual historical trajectory?' on arXiv...
URL: http://export.arxiv.org/api/query?search_query=all:If%20the%20Roman%20Empire%20had%20never%20fallen%2C%20how%20might%20the%20development%20of%20Western%20legal%20traditions%2C%20specifically%20concerning%20the%20rights%20of%20individuals%20versus%20the%20state%2C%20have%20differed%20from%20the%20actual%20historical%20trajectory%3F&start=0&max_results=100
Successfully retrieved search results!


In [17]:
ns = {"ns": "http://www.w3.org/2005/Atom"}
entries = parse_xml.findall('ns:entry', ns)

articles_data = []
for entry in entries:
    link = entry.find('ns:link[@type="application/pdf"]', ns)
    if link is not None and "href" in link.attrib:
        pdf_url = link.attrib['href']

        title = entry.find('ns:title', ns)
        title_text = title.text.strip() if title is not None else "Unknown Title"

        authors = entry.findall('ns:author/ns:name', ns)
        author_names = [author.text for author in authors] if authors else ["Unknown Author"]

        published = entry.find('ns:published', ns)
        published_date = published.text[:10] if published is not None else "Unknown Date"

        summary = entry.find('ns:summary', ns)
        summary_text = summary.text.strip() if summary is not None else "No summary available"

        metadata = {
            'title': title_text,
            'authors': author_names,
            'published': published_date,
            'summary': summary_text
        }

        articles_data.append({
            'pdf_url': pdf_url,
            'metadata': metadata
        })

print(f"Found {len(articles_data)} articles with PDF links")
for i, article in enumerate(articles_data):
    print(f"{i+1}. {article['metadata']['title'][:80]}...")

Found 100 articles with PDF links
1. Islamic Law, Western European Law and the Roots of Middle East's Long
  Divergen...
2. Redefining Accountability: Navigating Legal Challenges of Participant
  Liabilit...
3. Automated Refugee Case Analysis: An NLP Pipeline for Supporting Legal
  Practiti...
4. Understanding the Impact of Physicians' Legal Considerations on XAI
  Systems...
5. Towards A Structured Overview of Use Cases for Natural Language
  Processing in ...
6. Global recessions as a cascade phenomenon with heterogenous, interacting
  agent...
7. On Preemption and Overdetermination in Formal Theories of Causality...
8. Privacy Perspectives and Practices of Chinese Smart Home Product Teams...
9. Certifying and removing disparate impact...
10. Can AI be Consentful?...
11. Computer Modeling of Personal Autonomy and Legal Equilibrium...
12. CaseGNN: Graph Neural Networks for Legal Case Retrieval with
  Text-Attributed G...
13. Large Language Models as Fiduciaries: A Case Study Toward Ro

In [18]:
for article in articles_data:
    try:
        response = requests.get(article['pdf_url'])
        response.raise_for_status()

        pdf_reader = PdfReader(BytesIO(response.content))
        text_content = []

        for page in pdf_reader.pages:
            text_content.append(page.extract_text())

        full_text = "\n".join(text_content)

        if full_text:
            embedding = sentence_transformer_model.encode(full_text, convert_to_tensor=True)
            doc_index.add(np.array([embedding.numpy()]))
    except Exception as e:
        print(f"Error processing article: {e}")
        articles_data.remove(article)
        continue

Error processing article: 404 Client Error: Not Found for url: http://arxiv.org/pdf/2407.13070v2
Error processing article: 404 Client Error: Not Found for url: http://arxiv.org/pdf/2407.10329v1


In [19]:
query_vector = sentence_transformer_model.encode(question, convert_to_tensor=True)
query_vector = query_vector.reshape(1, -1)

k = 5
distances, indices = doc_index.search(query_vector, k)

In [20]:
print("Distances to nearest neighbors:", distances)
print("Indices of nearest neighbors:", indices)

Distances to nearest neighbors: [[1.091991  1.3660468 1.3731871 1.3743085 1.3793209]]
Indices of nearest neighbors: [[ 0 18 27 90 79]]


In [21]:
def chunk_text(text, max_chunk_length):
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        if current_length + len(word) + 1 <= max_chunk_length:
            current_chunk.append(word)
            current_length += len(word) + 1
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [word]
            current_length = len(word) + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [22]:
for index in indices[0]:
    article = articles_data[index]
    chunk_size = 1000
    response = requests.get(article['pdf_url'])
    response.raise_for_status()

    pdf_reader = PdfReader(BytesIO(response.content))
    text_content = []

    for page in pdf_reader.pages:
        text_content.append(page.extract_text())

    full_text = "\n".join(text_content)
    chunks = chunk_text(full_text, chunk_size)

    chunk_index.add(np.array([sentence_transformer_model.encode(chunk, convert_to_tensor=True).numpy() for chunk in chunks]))

In [23]:
query_vector = sentence_transformer_model.encode(question, convert_to_tensor=True)
query_vector = query_vector.reshape(1, -1)

k = 2
distances, indices = doc_index.search(query_vector, k)

In [24]:
print("Distances to nearest neighbors:", distances)
print("Indices of nearest neighbors:", indices)

Distances to nearest neighbors: [[1.091991  1.3660468]]
Indices of nearest neighbors: [[ 0 18]]


In [28]:
prompt = f"""
          You are an AI assistant to answer questions using context from specific chunks from journals.
          1 - read through the chunks
          2 - read through the question
          3 - answer the question using the chunks

          Context:
          {[chunks[index] for index in indices[0]]}

          Question:
          {question}
          """

In [29]:
print(prompt)


          You are an AI assistant to answer questions using context from specific chunks from journals.
          1 - read through the chunks
          2 - read through the question
          3 - answer the question using the chunks

          Context:
          ['Implications of Current Litigation on the Design of AI Systems for Healthcare Delivery Gennie Mansi, Mark Riedl Georgia Institute of Technology Atlanta, GA gennie.mansi@gatech.edu, riedl@cc.gatech.edu Abstract Many calls for explainable AI (XAI) systems in medicine are tied to a desire for AI accountability—accounting for, miti- gating, and ultimately preventing harms from AI systems. Be- cause XAI systems provide human-understandable explana- tions for their output, they are often viewed as a primary path to prevent harms to patients. However, when harm occurs, laws, policies, and regulations also shape AI accountability by impacting how harmed individuals can obtain recourse. Current approaches to XAI explore physicians’ m

In [ ]:
response_generate = client.generate(
        model=model_name,
        prompt=prompt
    )
print("Answer:", response_generate['response'])

ran it locally:

What an intriguing question!

To answer this, I'll draw upon the context provided. While there isn't a direct connection between the Roman Empire's fall and the development of Western legal traditions, we can analyze how the implications
of current litigation on AI systems might have influenced the trajectory.

In the context, it is mentioned that legal cases and reported harms shape AI accountability by impacting how harmed individuals can obtain recourse. This suggests that the development of legal frameworks,
including those concerning individual rights versus state power, has a significant impact on the way harms are addressed and compensated.

If the Roman Empire had never fallen, it's possible that the development of Western legal traditions would have been influenced by the empire's continued existence. The Roman Empire was known for its
codification of laws, with the Corpus Juris Civilis being a notable example. This might have led to a more centralized approach to law-making, potentially shaping the trajectory of individual rights versus
state power.

In this scenario, the development of Western legal traditions might have been characterized by:

1. Greater emphasis on imperial authority: The Roman Empire's continued dominance could have led to a stronger emphasis on imperial authority and less focus on individual rights.
2. More centralized law-making: The Corpus Juris Civilis would likely have remained a prominent influence, leading to more centralized law-making and potentially fewer opportunities for individual rights to
be recognized and protected.
3. Different approaches to compensation and deterrence: As mentioned in the context, deterrence and compensation often go hand-in-hand. In this scenario, the Roman Empire's continued dominance might have led
to different approaches to compensation and deterrence, with a greater emphasis on imperial interests rather than individual rights.

In conclusion, while it's impossible to know exactly how Western legal traditions would have developed if the Roman Empire had never fallen, it's likely that the trajectory would have been influenced by the
empire's continued existence. The development of laws and regulations would have been shaped by the empire's authority, potentially leading to a more centralized approach to law-making and a greater emphasis
on imperial interests over individual rights.


In [47]:
# Option 2 - Taking out stopwords
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')


sw_nltk = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

question = question.lower()
words = nltk.word_tokenize(question)
words_no_punct = [re.sub(r'[^\w\s]', '', word) for word in words]
words_no_punct = [word for word in words_no_punct if word]
filtered_words = [word for word in words_no_punct if word not in sw_nltk]
lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]
keyword = ' '.join(lemmatized_words)

print(f"Processed keyword: {keyword}")

Processed keyword: roman empire never fallen might development western legal tradition specifically concerning right individual versus state differed actual historical trajectory


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [48]:
num_articles = 100
encodingmethod = "utf-8"
errortype = "strict"

In [49]:
encoded_search_term = urllib.parse.quote(keyword, encoding=encodingmethod, errors=errortype)
url = f'http://export.arxiv.org/api/query?search_query=all:{encoded_search_term}&start=0&max_results={num_articles}'

print(f"Searching for '{keyword}' on arXiv...")
print(f"URL: {url}")

try:
    response = urllib.request.urlopen(url)
    try:
        url_read = response.read().decode("utf-8")
    except UnicodeDecodeError:
        response = urllib.request.urlopen(url)
        url_read = response.read().decode("utf-8", errors="ignore")

    parse_xml = ET.fromstring(url_read)
    print("Successfully retrieved search results!")
except Exception as e:
    print(f"Error retrieving data: {e}")
    raise

Searching for 'roman empire never fallen might development western legal tradition specifically concerning right individual versus state differed actual historical trajectory' on arXiv...
URL: http://export.arxiv.org/api/query?search_query=all:roman%20empire%20never%20fallen%20might%20development%20western%20legal%20tradition%20specifically%20concerning%20right%20individual%20versus%20state%20differed%20actual%20historical%20trajectory&start=0&max_results=100
Successfully retrieved search results!


In [50]:
ns = {"ns": "http://www.w3.org/2005/Atom"}
entries = parse_xml.findall('ns:entry', ns)

articles_data = []
for entry in entries:
    link = entry.find('ns:link[@type="application/pdf"]', ns)
    if link is not None and "href" in link.attrib:
        pdf_url = link.attrib['href']

        title = entry.find('ns:title', ns)
        title_text = title.text.strip() if title is not None else "Unknown Title"

        authors = entry.findall('ns:author/ns:name', ns)
        author_names = [author.text for author in authors] if authors else ["Unknown Author"]

        published = entry.find('ns:published', ns)
        published_date = published.text[:10] if published is not None else "Unknown Date"

        summary = entry.find('ns:summary', ns)
        summary_text = summary.text.strip() if summary is not None else "No summary available"

        metadata = {
            'title': title_text,
            'authors': author_names,
            'published': published_date,
            'summary': summary_text
        }

        articles_data.append({
            'pdf_url': pdf_url,
            'metadata': metadata
        })

print(f"Found {len(articles_data)} articles with PDF links")
for i, article in enumerate(articles_data):
    print(f"{i+1}. {article['metadata']['title'][:80]}...")

Found 100 articles with PDF links
1. Privacy Perspectives and Practices of Chinese Smart Home Product Teams...
2. Computer Modeling of Personal Autonomy and Legal Equilibrium...
3. Redefining Accountability: Navigating Legal Challenges of Participant
  Liabilit...
4. A Multi-solution Study on GDPR AI-enabled Completeness Checking of DPAs...
5. Islamic Law, Western European Law and the Roots of Middle East's Long
  Divergen...
6. Automated Refugee Case Analysis: An NLP Pipeline for Supporting Legal
  Practiti...
7. From the historical Roman road network to modern infrastructure in Italy...
8. A Temporal FRBR/FRBRoo-Based Model for Component-Level Versioning of
  Legal Nor...
9. On Preemption and Overdetermination in Formal Theories of Causality...
10. Examining the Legal Status of Digital Assets as Property: A Comparative
  Analys...
11. European historical evidence of the supernova of AD 1054 coins of
  Constantine ...
12. The Dictator Dilemma: The Distortion of Information Flow in Aut

In [55]:
for article in articles_data:
    try:
        response = requests.get(article['pdf_url'])
        response.raise_for_status()

        pdf_reader = PdfReader(BytesIO(response.content))
        text_content = []

        for page in pdf_reader.pages:
            text_content.append(page.extract_text())

        full_text = "\n".join(text_content)

        if full_text:
            embedding = sentence_transformer_model.encode(full_text, convert_to_tensor=True)
            doc_index.add(np.array([embedding.numpy()]))
    except Exception as e:
        print(f"Error processing article: {e}")
        articles_data.remove(article)
        continue

In [56]:
query_vector = sentence_transformer_model.encode(question, convert_to_tensor=True)
query_vector = query_vector.reshape(1, -1)

k = 5
distances, indices = doc_index.search(query_vector, k)

In [57]:
print("Distances to nearest neighbors:", distances)
print("Indices of nearest neighbors:", indices)

Distances to nearest neighbors: [[1.091991  1.3660468 1.3793209 1.3957342 1.4110671]]
Indices of nearest neighbors: [[ 4  7 93 83  6]]


In [58]:
for index in indices[0]:
    article = articles_data[index]
    chunk_size = 1000
    response = requests.get(article['pdf_url'])
    response.raise_for_status()

    pdf_reader = PdfReader(BytesIO(response.content))
    text_content = []

    for page in pdf_reader.pages:
        text_content.append(page.extract_text())

    full_text = "\n".join(text_content)
    chunks = chunk_text(full_text, chunk_size)

    chunk_index.add(np.array([sentence_transformer_model.encode(chunk, convert_to_tensor=True).numpy() for chunk in chunks]))

In [59]:
query_vector = sentence_transformer_model.encode(question, convert_to_tensor=True)
query_vector = query_vector.reshape(1, -1)

k = 2
distances, indices = doc_index.search(query_vector, k)

In [60]:
print("Distances to nearest neighbors:", distances)
print("Indices of nearest neighbors:", indices)

Distances to nearest neighbors: [[1.091991  1.3660468]]
Indices of nearest neighbors: [[4 7]]


In [61]:
prompt = f"""
          You are an AI assistant to answer questions using context from specific chunks from journals.
          1 - read through the chunks
          2 - read through the question
          3 - answer the question using the chunks

          Context:
          {[chunks[index] for index in indices[0]]}

          Question:
          {question}
          """

In [62]:
print(prompt)


          You are an AI assistant to answer questions using context from specific chunks from journals.
          1 - read through the chunks
          2 - read through the question
          3 - answer the question using the chunks

          Context:
          ["for welfare, equity and social inclusion, but some heterogeneity can exists at the subnational level. 1 that speci c local factors a ect the institution's functioning.3The idea at the heart of this work is that Roman roads have positively a ected current transport systems, regardless of the variety of historical paths within the Italian territory. This paper follows the strand of research that quanti es the long-term e ects of historical events on current development (Nunn, 2009), and the line of investigation of Temin (2013), Michaels and Rauch (2018), Wahl (2017), Dalgaard et al. (2018) and Flueckiger et al. (2022). The evidence here provided embraces two dimensions: the persistent e ect of history and the mechanism linkin

locally ran:
What an intriguing question!

To answer this, I'll draw upon the context provided. While there isn't a direct connection between the Roman Empire's fall and the development of Western legal traditions, we can analyze how the persistent
effects of historical events might have influenced the trajectory.

In the context, it is mentioned that the "persistent effect of history" has been studied in various aspects of the economy today. This strand of research explores the long-lasting impact of historical events
on modern economic development. If the Roman Empire had never fallen, it's possible that the development of Western legal traditions would have been influenced by the empire's continued existence.

One could argue that the absence of the fall of the Roman Empire might have led to a more centralized approach to law-making, potentially shaping the trajectory of individual rights versus state power. The
Roman Empire was known for its codification of laws, with the Corpus Juris Civilis being a notable example. This might have led to a stronger emphasis on imperial authority and less focus on individual
rights.

In this scenario, the development of Western legal traditions might have been characterized by:

1. Greater emphasis on imperial authority: The Roman Empire's continued dominance could have led to a stronger emphasis on imperial authority and less focus on individual rights.
2. More centralized law-making: The Corpus Juris Civilis would likely have remained a prominent influence, leading to more centralized law-making and potentially fewer opportunities for individual rights to
be recognized and protected.

However, it is important to note that this is purely speculative, as the actual historical trajectory of Western legal traditions was shaped by the fall of the Roman Empire and subsequent events.

In [ ]:
# Option 3 - Ask the LLM to generate good keywords from the question.
prompt = f"""Take the question and give the important search keywords to best find answers from the question.
          Question: {question}"""
prompt = f"""Take the question and give the important search keywords to best find answers from the question.
          Return all the keywords in a single line seperated by a comma. Dont say anything else. Here is an example: 'keyword1, keyword2, keyword3'
          Question: {question}
"""
response_generate = client.generate(
        model=model_name,
        prompt=prompt
    )
keyword = response_generate['response']

In [66]:
search_terms = "Roman Empire, Western legal traditions, individual rights vs. state power, historical trajectory, jurisprudence, law and politics, Roman law, medieval law, feudal law, early modern law, enlightenment era law"
search_terms = search_terms.lower()
words = nltk.word_tokenize(search_terms)
words_no_punct = [re.sub(r'[^\w\s]', '', word) for word in words]
words_no_punct = [word for word in words_no_punct if word]
filtered_words = [word for word in words_no_punct if word not in sw_nltk]
lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]
keyword = ' '.join(lemmatized_words)

In [67]:
keyword

'roman empire western legal tradition individual right v state power historical trajectory jurisprudence law politics roman law medieval law feudal law early modern law enlightenment era law'

In [68]:
num_articles = 100
encodingmethod = "utf-8"
errortype = "strict"

In [69]:
encoded_search_term = urllib.parse.quote(keyword, encoding=encodingmethod, errors=errortype)
url = f'http://export.arxiv.org/api/query?search_query=all:{encoded_search_term}&start=0&max_results={num_articles}'

print(f"Searching for '{keyword}' on arXiv...")
print(f"URL: {url}")

try:
    response = urllib.request.urlopen(url)
    try:
        url_read = response.read().decode("utf-8")
    except UnicodeDecodeError:
        response = urllib.request.urlopen(url)
        url_read = response.read().decode("utf-8", errors="ignore")

    parse_xml = ET.fromstring(url_read)
    print("Successfully retrieved search results!")
except Exception as e:
    print(f"Error retrieving data: {e}")
    raise

Searching for 'roman empire western legal tradition individual right v state power historical trajectory jurisprudence law politics roman law medieval law feudal law early modern law enlightenment era law' on arXiv...
URL: http://export.arxiv.org/api/query?search_query=all:roman%20empire%20western%20legal%20tradition%20individual%20right%20v%20state%20power%20historical%20trajectory%20jurisprudence%20law%20politics%20roman%20law%20medieval%20law%20feudal%20law%20early%20modern%20law%20enlightenment%20era%20law&start=0&max_results=100
Successfully retrieved search results!


In [70]:
ns = {"ns": "http://www.w3.org/2005/Atom"}
entries = parse_xml.findall('ns:entry', ns)

articles_data = []
for entry in entries:
    link = entry.find('ns:link[@type="application/pdf"]', ns)
    if link is not None and "href" in link.attrib:
        pdf_url = link.attrib['href']

        title = entry.find('ns:title', ns)
        title_text = title.text.strip() if title is not None else "Unknown Title"

        authors = entry.findall('ns:author/ns:name', ns)
        author_names = [author.text for author in authors] if authors else ["Unknown Author"]

        published = entry.find('ns:published', ns)
        published_date = published.text[:10] if published is not None else "Unknown Date"

        summary = entry.find('ns:summary', ns)
        summary_text = summary.text.strip() if summary is not None else "No summary available"

        metadata = {
            'title': title_text,
            'authors': author_names,
            'published': published_date,
            'summary': summary_text
        }

        articles_data.append({
            'pdf_url': pdf_url,
            'metadata': metadata
        })

print(f"Found {len(articles_data)} articles with PDF links")
for i, article in enumerate(articles_data):
    print(f"{i+1}. {article['metadata']['title'][:80]}...")

Found 5 articles with PDF links
1. Power laws in the Roman Empire: a survival analysis...
2. Human Indignity: From Legal AI Personhood to Selfish Memes...
3. Affirmative Algorithms: The Legal Grounds for Fairness as Awareness...
4. A Contextual Topic Modeling and Content Analysis of Iranian laws and
  Regulatio...
5. Analyzing Neural Scaling Laws in Two-Layer Networks with Power-Law Data
  Spectr...


In [73]:
for article in articles_data:
    try:
        response = requests.get(article['pdf_url'])
        response.raise_for_status()

        pdf_reader = PdfReader(BytesIO(response.content))
        text_content = []

        for page in pdf_reader.pages:
            text_content.append(page.extract_text())

        full_text = "\n".join(text_content)

        if full_text:
            embedding = sentence_transformer_model.encode(full_text, convert_to_tensor=True)
            doc_index.add(np.array([embedding.numpy()]))
    except Exception as e:
        print(f"Error processing article: {e}")
        articles_data.remove(article)
        continue

In [74]:
query_vector = sentence_transformer_model.encode(question, convert_to_tensor=True)
query_vector = query_vector.reshape(1, -1)

k = 5
distances, indices = doc_index.search(query_vector, k)

In [75]:
print("Distances to nearest neighbors:", distances)
print("Indices of nearest neighbors:", indices)

Distances to nearest neighbors: [[1.1200637 1.4245055 1.5120525 1.6014915 2.0284214]]
Indices of nearest neighbors: [[0 1 2 3 4]]


In [76]:
for index in indices[0]:
    article = articles_data[index]
    chunk_size = 1000
    response = requests.get(article['pdf_url'])
    response.raise_for_status()

    pdf_reader = PdfReader(BytesIO(response.content))
    text_content = []

    for page in pdf_reader.pages:
        text_content.append(page.extract_text())

    full_text = "\n".join(text_content)
    chunks = chunk_text(full_text, chunk_size)

    chunk_index.add(np.array([sentence_transformer_model.encode(chunk, convert_to_tensor=True).numpy() for chunk in chunks]))

In [77]:
query_vector = sentence_transformer_model.encode(question, convert_to_tensor=True)
query_vector = query_vector.reshape(1, -1)

k = 2
distances, indices = doc_index.search(query_vector, k)

In [78]:
print("Distances to nearest neighbors:", distances)
print("Indices of nearest neighbors:", indices)

Distances to nearest neighbors: [[1.1200637 1.4245055]]
Indices of nearest neighbors: [[0 1]]


In [79]:
prompt = f"""
          You are an AI assistant to answer questions using context from specific chunks from journals.
          1 - read through the chunks
          2 - read through the question
          3 - answer the question using the chunks

          Context:
          {[chunks[index] for index in indices[0]]}

          Question:
          {question}
          """

In [80]:
print(prompt)


          You are an AI assistant to answer questions using context from specific chunks from journals.
          1 - read through the chunks
          2 - read through the question
          3 - answer the question using the chunks

          Context:
          ['Preprint ANALYZING NEURAL SCALING LAWS IN TWO-LAYER NETWORKS WITH POWER -LAWDATA SPECTRA Roman Worschech Institut für Theoretische Physik Universität Leipzig Brüderstraße 16, 04103 Leipzig, Germany Max Planck Institute for Mathematics in the Sciences Inselstraße 22, 04103 Leipzig, Germany roman.worschech@uni-leipzig.deBernd Rosenow Institut für Theoretische Physik Universität Leipzig Brüderstraße 16, 04103 Leipzig, Germany ABSTRACT Neural scaling laws describe how the performance of deep neural networks scales with key factors such as training data size, model complexity, and training time, of- ten following power-law behaviors over multiple orders of magnitude. Despite their empirical observation, the theoretical understand

run locally:

This question is quite unrelated to the context provided, which appears to be about neural scaling laws in two-layer networks with power-law data spectra. The question asks about the hypothetical scenario
where the Roman Empire had never fallen and how it might have affected the development of Western legal traditions.

To answer this question, I'll need to draw upon my understanding of history, law, and society. If the Roman Empire had never fallen, it's likely that the development of Western legal traditions would have
been influenced by the empire's continued existence.

Here are some possible differences:

* The Roman Empire's codification of laws, as embodied in the Corpus Juris Civilis, might have remained a dominant influence on Western law for longer.
* The concept of individual rights versus state power might have developed differently, potentially with more emphasis on imperial authority and less focus on individual liberties.
* The fall of the Roman Empire led to the rise of feudalism in Europe, which had significant implications for the development of legal systems. In this hypothetical scenario, the absence of the fall of the
Roman Empire might have meant that feudalism would not have developed or might have been delayed.

These are just speculative ideas, and it's difficult to predict exactly how Western legal traditions would have developed without the fall of the Roman Empire.
